# Hierarchischer Ansatz: Ensemble

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** Prüfen, ob eine gewichtete Abstimmung aus RBF-SVM und den komplementärsten hierarchischen Klassifikatoren die SVM schlägt (#113).

In [ ]:
%load_ext autoreload
%autoreload 2

from itertools import combinations

import pandas as pd

from awp2.config import MIN_CV_GAIN, MODEL_DOCS_FIGURES_DIR
from awp2.data import combined_label, load_split
from awp2.evaluation import compare_predictions, error_breakdown
from awp2.experiment import RunConfig, TuneConfig, out_of_fold_predictions, run, tune
from awp2.models import CropStageVotingClassifier
from awp2.models.tuned import (
    refit_untracked,
    tuned_hierarchical_extra_trees,
    tuned_hierarchical_knn,
    tuned_hierarchical_logistic_regression,
    tuned_hierarchical_svm,
)
from awp2.plots import plot_cv_scores, plot_error_breakdown, save_doc_figure

In [ ]:
split = load_split()
split.X_train.shape

## Hypothese

Die hierarchischen Klassifikatoren ergänzen sich: Auf dem Trainingsteil (out-of-fold) trifft
mindestens einer 96,2 % der Zeilen, die RBF-SVM allein 87,7 % (#104). Am häufigsten treffen
kNN, logistische Regression und Extra Trees die Kulturfehler der SVM – die Kandidaten wurden dort
auf dem Trainingsteil bestimmt, nicht auf der Validierung.

`CropStageVotingClassifier` lässt jedes Mitglied ein vollständiges Crop/Stage-Paar vorhersagen;
das Paar mit den meisten (gewichteten) Stimmen gewinnt, bei Gleichstand die SVM. `first_weight`
ist das Gewicht der SVM. Weil Gleichstände an die SVM gehen, entscheiden Gewicht 1 und 1.5
gleich, und bei zwei Mitgliedern oder Gewicht 2 mit bis zu drei Mitgliedern ist das Ensemble die
SVM – die 21 Kandidaten sind nur etwa 5 verschiedene Entscheidungsregeln. Alle Mitglieder behalten ihre getunten Einstellungen (`awp2.models.tuned`) und
dieselbe Vorverarbeitung.

Hinweis: Die Mitglieder wurden auf denselben CV-Folds getunt; der CV-Score des Ensembles ist
daher – wie der jeder Einzelsuche – leicht optimistisch, der Vergleich mit der SVM aber fair.

In [ ]:
reference = tuned_hierarchical_svm()
complements = {
    "knn": tuned_hierarchical_knn(),
    "logreg": tuned_hierarchical_logistic_regression(),
    "extra_trees": tuned_hierarchical_extra_trees(),
}
member_sets = [
    [("svm", reference.model)] + [(name, complements[name].model) for name in chosen]
    for size in range(1, len(complements) + 1)
    for chosen in combinations(complements, size)
]
len(member_sets), reference.cv_score

## CV-Suche: Mitglieder und Gewicht der SVM

In [ ]:
tuned = tune(
    CropStageVotingClassifier(member_sets[0]),
    TuneConfig(
        name="hierarchical_ensemble_search",
        approach="hierarchical",
        description="Gewichtete Abstimmung: RBF-SVM plus jede Teilmenge aus kNN, logistischer "
        "Regression und Extra Trees (getunte Einstellungen); Gewicht der SVM 1, 1.5 oder 2.",
        preprocessing=reference.preprocessing,
        param_grid={"estimators": member_sets, "first_weight": [1.0, 1.5, 2.0]},
    ),
)


def members(params: dict) -> str:
    return " + ".join(name for name, _ in params["estimators"])


candidates = tuned.candidates.assign(
    Mitglieder=[members(p) for p in tuned.candidates["params"]],
    SVM_Gewicht=[p["first_weight"] for p in tuned.candidates["params"]],
)[["Mitglieder", "SVM_Gewicht", "mean", "std", "rank"]].sort_values("rank")
candidates.round(4)

In [ ]:
by_members = {
    label: tuned.candidates[[members(p) == label for p in tuned.candidates["params"]]]
    for label in dict.fromkeys(candidates["Mitglieder"])
}
ensemble_ax = plot_cv_scores(by_members, reference=reference.cv_score)
ensemble_ax.set_title("Ensemble: bestes SVM-Gewicht je Mitgliedermenge (gestrichelt: SVM allein)")
save_doc_figure(ensemble_ax.figure, "hierarchical_ensemble_tuning", MODEL_DOCS_FIGURES_DIR)

## Gegenprobe auf dem Trainingsteil

Out-of-fold-Vorhersagen des besten Ensembles und der SVM: Woher kommt ein Gewinn – Kultur oder
Stadium? Nur Trainingsdaten.

In [ ]:
oof_config = RunConfig(
    name="hierarchical_ensemble_oof",
    approach="hierarchical",
    preprocessing=reference.preprocessing,
    track=False,
)
oof = {
    "RBF-SVM": out_of_fold_predictions(reference.model, oof_config),
    "Ensemble": out_of_fold_predictions(tuned.best_model, oof_config),
}
(
    pd.DataFrame(
        {label: error_breakdown(split.y_train, pred).model_dump() for label, pred in oof.items()}
    ).round(4),
    compare_predictions(split.y_train, oof["RBF-SVM"], oof["Ensemble"]),
)

Warum gewinnt die Abstimmung kaum, obwohl sich die Modelle ergänzen? Gezählt auf dem
Trainingsteil: Wo die SVM falsch liegt, wie viele der anderen Mitglieder haben recht?

In [ ]:
member_oof = {
    name: out_of_fold_predictions(tuned_member.model, oof_config)
    for name, tuned_member in complements.items()
}
true_pair = combined_label(split.y_train)
svm_wrong = combined_label(oof["RBF-SVM"]) != true_pair
right_others = pd.DataFrame(
    {name: combined_label(pred) == true_pair for name, pred in member_oof.items()}
).sum(axis=1)
ensemble_right = combined_label(oof["Ensemble"]) == true_pair
any_of_four = ~svm_wrong | (right_others > 0)
pd.Series(
    {
        "SVM falsch": svm_wrong.sum(),
        "davon mind. ein anderes Mitglied richtig": (svm_wrong & (right_others > 0)).sum(),
        "davon genau eines richtig": (svm_wrong & (right_others == 1)).sum(),
        "davon Ensemble richtig": (svm_wrong & ensemble_right).sum(),
        "mind. eines der vier richtig (Anteil)": round(any_of_four.mean(), 3),
    }
)

## Einmalige Bewertung

Nur wenn das beste Ensemble die SVM in der CV um mindestens `MIN_CV_GAIN` schlägt, wird es einmal
mit `run()` bewertet; sonst bleibt `hierarchical_svm` die Referenz.

In [ ]:
cv_gain = tuned.best_score - reference.cv_score
if cv_gain >= MIN_CV_GAIN:
    result = run(
        tuned.best_model,
        RunConfig(
            name="hierarchical_ensemble",
            approach="hierarchical",
            description=f"Bestes Ensemble ({members(tuned.best_params)}, SVM-Gewicht "
            f"{tuned.best_params['first_weight']}) aus hierarchical_ensemble_search.",
            preprocessing=reference.preprocessing,
            tuning_run=tuned.run_id,
        ),
    )
    decision = result.metrics.model_dump()
else:
    result = None
    decision = {"Entscheidung": "SVM bleibt Referenz", "CV-Gewinn": round(cv_gain, 4)}
members(tuned.best_params), tuned.best_params["first_weight"], round(cv_gain, 4), decision

In [ ]:
if result is not None:
    svm_result = refit_untracked(reference, "hierarchical_svm_refit")
    predictions = {"RBF-SVM": svm_result.y_pred, "Ensemble": result.y_pred}
    breakdowns = {label: error_breakdown(split.y_val, pred) for label, pred in predictions.items()}
    breakdown_ax = plot_error_breakdown(breakdowns)
    breakdown_ax.set_title("Fehlerzerlegung: Ensemble vs. RBF-SVM (Validierung)")
    save_doc_figure(breakdown_ax.figure, "hierarchical_ensemble_breakdown", MODEL_DOCS_FIGURES_DIR)
    validation = (
        pd.DataFrame(
            {
                label: {**r.metrics.model_dump(), **breakdowns[label].model_dump()}
                for label, r in {"RBF-SVM": svm_result, "Ensemble": result}.items()
            }
        ).round(4),
        compare_predictions(split.y_val, svm_result.y_pred, result.y_pred),
    )
else:
    validation = "keine Bewertung"
validation

## Befund

- Bestes Ensemble: alle vier Modelle mit gleichem Gewicht, CV 0.844 gegen 0.839 der SVM – Gewinn
  0.0045 unter `MIN_CV_GAIN`, daher keine Validierung; die SVM bleibt Referenz. Gewicht 1 und 1.5
  entscheiden gleich (Gleichstand geht an die SVM).
- Auf dem Trainingsteil (out-of-fold) kaum Änderung: Kultur falsch 8,15 % (SVM) vs. 8,13 %;
  der kleine Gewinn kommt vom Stadium. McNemar p = 0.31.
- Mindestens eines der vier trifft 94,9 % der Zeilen, aber von 282 SVM-Fehlern, die ein anderes
  Mitglied trifft, trifft es in 150 nur genau eines – das unterliegt in der Abstimmung; das
  Ensemble rettet 54.
- Ein erster Lauf mit fehlerhafter Gleichstandsregel (alphabetisch statt zugunsten der SVM) ist
  ungültig (`d4e93003`).